# Day 7: Incremental Processing & CDC Upsert Engine

This notebook simulates idempotent UPSERT/DELETE logic on a stateful Iceberg table using MERGE INTO.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp

spark = SparkSession.builder \
    .appName("Day07_Incremental_Upsert") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse_d7") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip")
spark.sql("DROP TABLE IF EXISTS local.ehdip.silver_patients")

spark.sql("""
CREATE TABLE local.ehdip.silver_patients (
    id INT,
    name STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.ehdip.silver_patients VALUES
(1, 'Alice', timestamp '2023-01-01 10:00:00'),
(2, 'Bob', timestamp '2023-01-01 10:00:00')
""")

print("Initial State:")
spark.sql("SELECT * FROM local.ehdip.silver_patients ORDER BY id").show()

In [ ]:
# Simulate CDC Updates
updates_data = [
    (1, 'Alice Updated', '2023-01-02 10:00:00', 'u'), # Valid update
    (2, 'Bob Stale', '2022-12-31 10:00:00', 'u'),     # Out of order update (should be ignored)
    (3, 'Charlie', '2023-01-02 11:00:00', 'c'),       # New insert
]
updates_df = spark.createDataFrame(updates_data, ["id", "name", "updated_at_str", "_cdc_op"]) \
    .withColumn("updated_at", to_timestamp("updated_at_str")) \
    .drop("updated_at_str")

updates_df.createOrReplaceTempView("updates")

print("Incoming CDC Updates:")
updates_df.show()

In [ ]:
merge_sql = """
MERGE INTO local.ehdip.silver_patients t
USING updates u
ON t.id = u.id
WHEN MATCHED AND u._cdc_op = 'd' THEN
    DELETE
WHEN MATCHED AND u.updated_at > t.updated_at THEN
    UPDATE SET t.name = u.name, t.updated_at = u.updated_at
WHEN NOT MATCHED AND u._cdc_op != 'd' THEN
    INSERT (id, name, updated_at) VALUES (u.id, u.name, u.updated_at)
"""

spark.sql(merge_sql)

print("Final State after MERGE:")
spark.sql("SELECT * FROM local.ehdip.silver_patients ORDER BY id").show()